# DINOv2

#### imports

In [7]:
! python -m pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.1 MB 15.0 MB/s            
     |████████████████████████████████| 70 kB 3.4 MB/s             
     |████████████████████████████████| 11.3 MB 121.8 MB/s            
     |████████████████████████████████| 98 kB 2.8 MB/s             
     |████████████████████████████████| 323 kB 51.8 MB/s            
     |████████████████████████████████| 64 kB 1.0 MB/s             
     |████████████████████████████████| 212 kB 66.5 MB/s            
     |████████████████████████████████| 66 kB 2.2 MB/s             
     |████████████████████████████████| 731 kB 63.1 MB/s            
     |████████████████████████████████| 12.8 MB 76.2 MB/s            
     |████████████████████████████████| 42.7 MB 40.6 MB/s            
     |████████████████████████████████| 79 kB 3.2 MB/s             
     |████████████████████████████████| 62.5 MB 113 kB/s             
     |██████████████████

In [8]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import AutoModel, AutoImageProcessor
import yaml
from collections import defaultdict
import glob
from torch.utils.data import Dataset, DataLoader
from peft import get_peft_model, LoraConfig

### fully-frozen DINO

In [ ]:
'''

This cell extracts DINOv2 embeddings for each object in the Roboflow dataset (YOLOv8 format),
using the provided YOLO labels to crop objects from images. It saves the resulting embeddings,
labels, and source image names to a .npz file for later use in k-NN classification.

'''

# source: https://medium.com/data-science-in-your-pocket/getting-started-with-dinov2-installation-setup-and-inference-made-easy-be37b07d32e7
# https://huggingface.co/docs/transformers/en/model_doc/dinov2

DATASET_DIR = "../ingredients_data" # dataset with images and labels (YOLOv8 format)
SPLIT = "train"

IMG_DIR   = os.path.join(DATASET_DIR, SPLIT, "images") # path for images
LABEL_DIR = os.path.join(DATASET_DIR, SPLIT, "labels") # path for labels

OUTPUT_FILE = f"dinov2_objects_{SPLIT}.npz" # path of output file

MODEL_NAME = "facebook/dinov2-base" # using dinov2 for object embedding extraction
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # use gpu if available


# load pretrained model dinov2
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

for p in model.parameters(): # no need for gradients, just inference
    p.requires_grad = False


# parse a line in the label file
def parse_label_line(line): # get class and box from a line in the label file
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    class_id = int(parts[0])
    x_center, y_center, w, h = [float(x) for x in parts[1:5]]

    return class_id, (x_center, y_center, w, h)


# load labels for an image
def load_labels(image_filename):
    # get equivalent label file path for a given image file
    label_path = os.path.join(LABEL_DIR, os.path.splitext(image_filename)[0] + ".txt")

    if not os.path.exists(label_path):
        return []

    labels = []
    with open(label_path, "r") as f:
        for line in f:
            parsed = parse_label_line(line)
            if parsed:
                labels.append(parsed)

    return labels


# convert yolo format from labels, to pixels
def yolo_to_coords(box, img_w, img_h):
    # source: https://www.geeksforgeeks.org/python/python-pil-image-crop-method/
    # yolo format is (x_center, y_center, width, height) normalized to [0,1]
    # .crop() requires (left, top, right, bottom)
    x, y, w, h = box

    left = int((x - w / 2) * img_w) # * img_w to get pixel coord from normalized
    top = int((y - h / 2) * img_h)
    right = int((x + w / 2) * img_w)
    bottom = int((y + h / 2) * img_h)

    return left, top, right, bottom


# embeded crop using dinov2
def embed_crop(crop_img):
    inputs = processor(images=crop_img, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    emb = out.last_hidden_state[:, 0, :].squeeze()
    return emb.cpu().numpy()


# process Roboflow dataset
embeddings = []
labels = []
sources = []

image_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

print(f"Found {len(image_files)} images")

print("\nExtracting object embeddings\n")

# run for every image
for fname in image_files:
    img_path = os.path.join(IMG_DIR, fname)

    try:
        img = Image.open(img_path).convert("RGB")
    except:
        continue

    img_w, img_h = img.size

    yolo_labels = load_labels(fname)

    for class_id, box in yolo_labels:
        # get pixel coords from yolo format labels
        left, top, right, bottom = yolo_to_coords(box, img_w, img_h)

        x1, y1 = max(0, left), max(0, top)
        x2, y2 = min(img_w, right), min(img_h, bottom)

        if x2 <= x1 or y2 <= y1:
            continue

        # crop the image to the bounding box of the object
        crop = img.crop((x1, y1, x2, y2))

        # get embedding for the cropped object
        emb = embed_crop(crop)

        embeddings.append(emb)
        labels.append(class_id)
        sources.append(fname)


embeddings = np.stack(embeddings)

embeddings /= (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8) # normalization

labels = np.array(labels)
sources = np.array(sources)


# save to npz
np.savez(
    OUTPUT_FILE,
    embeddings=embeddings,
    labels=labels,
    sources=sources
)

print(f"Saved to {OUTPUT_FILE}")

accuracy: 85.3%

### LoRA

In [ ]:
'''

Fine-tune DINOv2 using LoRA (Low-Rank Adaptation) via `peft` library.

'''

# source: https://huggingface.co/docs/peft/main/en/conceptual_guides/lora

DATASET_DIR = "../ingredients_data"
SPLIT = "train"

IMG_DIR = os.path.join(DATASET_DIR, SPLIT, "images")
LABEL_DIR = os.path.join(DATASET_DIR, SPLIT, "labels")
OUTPUT_FILE = f"dinov2_objects_{SPLIT}.npz"

MODEL_NAME = "facebook/dinov2-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4

print(f"Using device: {DEVICE}")

# get class names

with open(os.path.join(DATASET_DIR, "data.yaml"), "r") as f:
    CLASS_NAMES = yaml.safe_load(f)["names"]
num_classes = len(CLASS_NAMES)
print(f"Classes ({num_classes}): {CLASS_NAMES}")

# LoRA configuration

lora_config = LoraConfig(
    r=8, # rank (controls the number of updated matrices and trainable parameters)
    lora_alpha=8, # LoRA scaling factor
    lora_dropout=0.1, # dropout to prevent overfitting
    target_modules=["query", "value"] # which modules to apply LoRA to
)

PROCESSOR = AutoImageProcessor.from_pretrained(MODEL_NAME)
backbone  = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
backbone  = get_peft_model(backbone, lora_config)
backbone.print_trainable_parameters()

# classification head (on top of LoRA-modified backbone)
embed_dim  = backbone.config.hidden_size
classifier = nn.Linear(embed_dim, num_classes).to(DEVICE)


def parse_label_line(line):
    parts = line.strip().split()
    if len(parts) < 5:
        return None
    class_id = int(parts[0])
    x_center, y_center, w, h = [float(x) for x in parts[1:5]]
    return class_id, (x_center, y_center, w, h)

def load_labels(image_filename):
    label_path = os.path.join(LABEL_DIR, os.path.splitext(image_filename)[0] + ".txt")
    if not os.path.exists(label_path):
        return []
    labels = []
    with open(label_path, "r") as f:
        for line in f:
            parsed = parse_label_line(line)
            if parsed:
                labels.append(parsed)
    return labels

def yolo_to_coords(box, img_w, img_h):
    x, y, w, h = box
    left = int((x - w / 2) * img_w)
    top = int((y - h / 2) * img_h)
    right = int((x + w / 2) * img_w)
    bottom = int((y + h / 2) * img_h)
    return left, top, right, bottom


class CropDataset(Dataset):
    def __init__(self):
        self.samples   = [] # to store (crop, class_id) pairs
        self.processor = PROCESSOR

        image_files = [f for f in os.listdir(IMG_DIR)
                       if f.lower().endswith((".jpg", ".png", ".jpeg"))]
        print(f"Building dataset from {len(image_files)} images...")

        for fname in tqdm(image_files):
            try:
                img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
            except Exception:
                continue
            img_w, img_h = img.size
            yolo_labels = load_labels(fname)
            for class_id, box in yolo_labels:
                left, top, right, bottom = yolo_to_coords(box, img_w, img_h)
                x1, y1 = max(0, left), max(0, top)
                x2, y2 = min(img_w, right), min(img_h, bottom)
                if x2 <= x1 or y2 <= y1:
                    continue
                self.samples.append((img.crop((x1, y1, x2, y2)), class_id))

        print(f"Total crops: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        crop, class_id = self.samples[idx]
        pixel_values = self.processor(images=crop, return_tensors="pt")["pixel_values"].squeeze(0)
        return pixel_values, class_id


# wrap dataset in DataLoader for batching
dataset = CropDataset(PROCESSOR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


optimizer = torch.optim.AdamW([
    {"params": backbone.parameters(), "lr": LR},
    {"params": classifier.parameters(), "lr": LR * 10},
])
loss_fn = nn.CrossEntropyLoss()

# training loop

print("\nTraining with LoRA...\n")

for epoch in range(NUM_EPOCHS):
    backbone.train()
    classifier.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for pixel_values, labels_batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        pixel_values = pixel_values.to(DEVICE)
        labels_batch = labels_batch.to(DEVICE)

        out = backbone(pixel_values=pixel_values)
        emb = out.last_hidden_state[:, 0, :] # CLS token

        logits = classifier(emb)
        loss = loss_fn(logits, labels_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == labels_batch).sum().item() # get highest logit index and compare to label for accuracy
        total += labels_batch.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total * 100
    print(f"Epoch {epoch+1:>3}/{NUM_EPOCHS}  loss: {avg_loss:.4f}  acc: {accuracy:.1f}%")

print("\nTraining complete.")

# save LoRA adapters and classifier head

backbone.save_pretrained("dinov2_lora_adapters/") # save LoRA weights injected into the backbone
torch.save(classifier.state_dict(), "dinov2_lora_classifier.pt") # save linear classifier head weights (maps backbone embeddings to class scores)
print("LoRA adapters saved to dinov2_lora_adapters/")
print("Classifier saved to dinov2_lora_classifier.pt")

# normalize embedding and save for kNN evaluation

print("\nExtracting embeddings for kNN...")

backbone.eval()
classifier.eval()

all_embeddings = []
all_labels     = []

for pixel_values, labels_batch in tqdm(dataloader, desc="Extracting"):
    pixel_values = pixel_values.to(DEVICE)
    with torch.no_grad():
        out = backbone(pixel_values=pixel_values)
    emb = out.last_hidden_state[:, 0, :].cpu().numpy()
    all_embeddings.append(emb)
    all_labels.extend(labels_batch.numpy())

embeddings = np.concatenate(all_embeddings, axis=0)
embeddings /= (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8)
labels = np.array(all_labels)
sources = np.array([""] * len(labels))

np.savez(OUTPUT_FILE, embeddings=embeddings, labels=labels, sources=sources)
print(f"Embeddings saved to {OUTPUT_FILE}")

LoRA configuration: rank=8, alpha=8, dropout=0.1

accuracy: 88.7%

## Interpretation

### kNN

In [ ]:
train = np.load("dinov2_objects_train.npz", allow_pickle=True)
test  = np.load("dinov2_objects_test.npz",  allow_pickle=True)

train_emb    = train["embeddings"]
train_labels = train["labels"]
train_sources = train["sources"]

test_emb     = test["embeddings"]
test_labels  = test["labels"]
test_sources = test["sources"]

with open("./dataset/data.yaml", "r") as f: # get class names from roboflow dataset yaml file
    CLASS_NAMES = yaml.safe_load(f)["names"]

output_lines = []

# to print and save output in a file at the end
def log(line=""):
    print(line)
    output_lines.append(line)

correct = 0

# kNN using cosine similarity
K = 5
for i in range(len(test_emb)):
    sims = train_emb @ test_emb[i] # cosine similarity between test embedding and all train embeddings
    # .argsort() gets indices of lowest to highest similarity
    # get k highest similarities (the last k)
    # reverse for the right order
    topk = sims.argsort()[-K:][::-1] # get indices of top k most similar train embeddings

    # majority vote among top k neighbors: get how many times each class appears and get the most common
    pred = np.bincount(train_labels[topk]).argmax()
    
    true_label = test_labels[i]
    is_correct = pred == true_label

    if is_correct:
        correct += 1

    log(f"[{'TRUE' if is_correct else 'FALSE'}] {CLASS_NAMES[true_label]} → predicted {CLASS_NAMES[pred]} | {test_sources[i]}")
    for rank, idx in enumerate(topk, 1):
        log(f"  {rank}. {CLASS_NAMES[train_labels[idx]]:<20} score: {sims[idx]:.3f}  ({train_sources[idx]})")
    log()


log(f"Total: {len(test_emb)}")
log(f"Correct: {correct}")
log(f"Wrong: {len(test_emb) - correct}")
log(f"Accuracy: {correct / len(test_emb) * 100:.1f}%")


with open("knn_results.txt", "w") as f:
    f.write("\n".join(output_lines))

print("\nSaved to knn_results.txt")

### test on a single image

In [ ]:
from peft import PeftModel
from pathlib import Path

LORA_ROOT = Path.cwd() /"dino"/ "lora" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_DIR = "./ingredients_data"

with open(os.path.join(DATASET_DIR, "data.yaml"), "r") as f:
    CLASS_NAMES = yaml.safe_load(f)["names"]
num_classes = len(CLASS_NAMES)

MODEL_NAME  = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained("facebook/dinov2-base").to(DEVICE)
backbone = PeftModel.from_pretrained(backbone, str(LORA_ROOT/ "dinov2_lora_adapters/"))

embed_dim = backbone.config.hidden_size
classifier = nn.Linear(embed_dim, num_classes).to(DEVICE)
classifier.load_state_dict(torch.load(LORA_ROOT / "dinov2_lora_classifier.pt", map_location=DEVICE))

backbone.eval()
classifier.eval()

print("num classes in head:", classifier.out_features)
print("num class names:", len(CLASS_NAMES))
print("first 10 class names:", CLASS_NAMES[:10])

# run on a single image
img = Image.open("fridge_data/t6.png").convert("RGB")

pixel_values = processor(images=img, return_tensors="pt")["pixel_values"].to(DEVICE)

with torch.no_grad():
    emb = backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :]
    logits = classifier(emb)
    probs = torch.softmax(logits, dim=1).squeeze()

pred_idx = probs.argmax().item()
pred_name = CLASS_NAMES[pred_idx]
pred_conf = probs[pred_idx].item()

print(f"Prediction : {pred_name}")
print(f"Confidence : {pred_conf:.1%}")
print()
print("All class probabilities:")
for i, (name, prob) in enumerate(zip(CLASS_NAMES, probs.tolist())):
    bar = "█" * int(prob * 30)
    print(f"  {name:<20} {prob:.1%}  {bar}")

### using YOLO box detection

In [ ]:
import json

def parse_json(jsonl_path):
    records = {}
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            records[Path(row["image"]).name] = row.get("true_ingredients", [])
    return records


def classification_accuracy(pred_ingredients, actual_ingredients):
    """
    Given detected ingredients and ground truth ingredients,
    compute per-image precision/recall/F1 and macro averages.
    """
    per_image = {}
    total_tp = total_fp = total_fn = 0

    for image_name, preds in pred_ingredients.items():
        pred_set = {p["ingredient"].lower() for p in preds}
        print(f"Predicted set: {pred_set}")
        true_set = set(i.lower() for i in actual_ingredients.get(image_name, []))
        print(f"True set: {true_set}")

        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        per_image[image_name] = {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
        }
        total_tp += tp
        total_fp += fp
        total_fn += fn

    global_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    global_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    global_f1 = (
        2 * global_precision * global_recall / (global_precision + global_recall)
        if (global_precision + global_recall)
        else 0.0
    )

    summary = {
        "global_precision": round(global_precision, 4),
        "global_recall": round(global_recall, 4),
        "global_f1": round(global_f1, 4),
        "images_evaluated": len(per_image),
    }

    return {"summary": summary, "per_image": per_image}

In [ ]:
import importlib
import yolo.yolo as yolo_mod

importlib.reload(yolo_mod)

best_checkpoint = yolo_mod.REPO_ROOT / "yolo"/ "runs" / "yolo11n_fridge-2" / "weights" / "best.pt"
model = yolo_mod.YOLO(str(best_checkpoint))
yolo_mod.test_model(model, dataset_path=yolo_mod.REPO_ROOT / "eval_data" / "images")

In [ ]:
from pathlib import Path
import sys
import cv2
import pandas as pd
from collections import defaultdict
from ultralytics import YOLO
from peft import PeftModel

# Ensure repo root import path for `from yolo.yolo import test_model`
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from yolo.yolo import test_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_DIR = REPO_ROOT / "ingredients_data"
MODEL_NAME = "facebook/dinov2-base"
LORA_ROOT = REPO_ROOT / "dino" / "lora"

with open(DATASET_DIR / "data.yaml", "r") as f:
    CLASS_NAMES = yaml.safe_load(f)["names"]
num_classes = len(CLASS_NAMES)

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
backbone = PeftModel.from_pretrained(backbone, str(LORA_ROOT / "dinov2_lora_adapters"))
embed_dim = backbone.config.hidden_size
classifier = nn.Linear(embed_dim, num_classes).to(DEVICE)
classifier.load_state_dict(torch.load(LORA_ROOT / "dinov2_lora_classifier.pt", map_location=DEVICE))

backbone.eval()
classifier.eval()

# Load YOLO checkpoint
yolo_model = YOLO(str(REPO_ROOT / "yolo"/ "runs" / "yolo11n_fridge-2" / "weights" / "best.pt"))

# Run on ALL eval images (directory)
EVAL_IMAGES_DIR = REPO_ROOT / "eval_data" / "images"
yolo_pred_ingredients = test_model(yolo_model, dataset_path=EVAL_IMAGES_DIR)

crops_dir = REPO_ROOT / "yolo" / "runs" / "eval" / "crops"
crop_files = sorted(crops_dir.glob("*.jpg"))
if not crop_files:
    raise FileNotFoundError(f"No crop images found in {crops_dir}. Run YOLO test_model first.")

dino_best_by_image = defaultdict(dict)
per_crop_rows = []

for crop_path in crop_files:
    # Getting origin image name
    image_stem = crop_path.stem.rsplit("_", 1)[0]

    crop_bgr = cv2.imread(str(crop_path))
    if crop_bgr is None:
        continue

    # Prepare images for DINO: color and format 
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB) 
    crop_pil = Image.fromarray(crop_rgb)
    pixel_values = processor(images=crop_pil, return_tensors="pt")["pixel_values"].to(DEVICE) 

    # Run Inference
    with torch.no_grad():
        emb = backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :]
        logits = classifier(emb)
        probs = torch.softmax(logits, dim=1).squeeze()

    pred_idx = probs.argmax().item()
    dino_label = CLASS_NAMES[pred_idx]
    dino_conf = float(probs[pred_idx].item())

    dino_key = dino_label.lower()
    dino_best_by_image[image_stem][dino_key] = max(dino_best_by_image[image_stem].get(dino_key, 0.0), dino_conf)

    per_crop_rows.append(
        {
            "image_stem": image_stem,
            "crop_file": crop_path.name,
            "dino_label": dino_label,
            "dino_conf": round(dino_conf, 4),
        }
    )

# Correct formatting to match that of the list outputed from test_model in yolo.py for metrics
dino_pred_ingredients = {}
for image_stem, cls_to_conf in dino_best_by_image.items():
    dino_pred_ingredients[f"{image_stem}.jpg"] = [
        {"ingredient": cls, "confidence": round(conf, 3)}
        for cls, conf in sorted(cls_to_conf.items(), key=lambda kv: kv[1], reverse=True)
    ]

actual_ingredients = parse_json(REPO_ROOT / "eval_data" / "labels.jsonl")
yolo_metrics = classification_accuracy(yolo_pred_ingredients, actual_ingredients)
dino_metrics = classification_accuracy(dino_pred_ingredients, actual_ingredients)

print("YOLO summary:", yolo_metrics["summary"])
print("DINO summary:", dino_metrics["summary"])

out_dir = REPO_ROOT / "yolo" / "runs" / "eval"
out_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(per_crop_rows).to_csv(out_dir / "dino_per_crop_predictions.csv", index=False)
pd.DataFrame([
    {"model": "yolo", **yolo_metrics["summary"]},
    {"model": "dino", **dino_metrics["summary"]},
]).to_csv(out_dir / "yolo_vs_dino_summary.csv", index=False)

print(f"Saved per-crop DINO predictions to {out_dir / 'dino_per_crop_predictions.csv'}")
print(f"Saved summary metrics to {out_dir / 'yolo_vs_dino_summary.csv'}")


In [ ]:
# YOLO vs DINO on every crop in yolo/runs/eval/crops (requires crop_manifest.json)
from eval_yolo_dino_crops import REPO_ROOT, compare_yolo_dino_on_crops

compare_yolo_dino_on_crops(crops_dir=REPO_ROOT / "yolo" / "runs" / "eval" / "crops", verbose=True)

___

# Extra

### Frozen DINOv2 with chunking for memory efficiency on bigger datasets

In [ ]:
'''

This cell extracts DINOv2 embeddings for each object in the Roboflow dataset (YOLOv8 format),
using the provided YOLO labels to crop objects from images. It saves the resulting embeddings,
labels, and source image names to a .npz file for later use in k-NN classification.

'''

# source: https://medium.com/data-science-in-your-pocket/getting-started-with-dinov2-installation-setup-and-inference-made-easy-be37b07d32e7
# https://huggingface.co/docs/transformers/en/model_doc/dinov2

DATASET_DIR = "./ingredients_data" # dataset with images and labels (YOLOv8 format)
SPLIT = "test"

IMG_DIR   = os.path.join(DATASET_DIR, SPLIT, "images") # path for images
LABEL_DIR = os.path.join(DATASET_DIR, SPLIT, "labels") # path for labels

MODEL_NAME = "facebook/dinov2-base" # using dinov2 for object embedding extraction
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # use gpu if available

chunk_id = 0

# load pretrained model dinov2
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

for p in model.parameters(): # no need for gradients, just inference
    p.requires_grad = False


# parse a line in the label file
def parse_label_line(line): # get class and box from a line in the label file
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    class_id = int(parts[0])
    x_center, y_center, w, h = [float(x) for x in parts[1:5]]

    return class_id, (x_center, y_center, w, h)


# load labels for an image
def load_labels(image_filename):
    # get equivalent label file path for a given image file
    label_path = os.path.join(LABEL_DIR, os.path.splitext(image_filename)[0] + ".txt")

    if not os.path.exists(label_path):
        return []

    labels = []
    with open(label_path, "r") as f:
        for line in f:
            parsed = parse_label_line(line)
            if parsed:
                labels.append(parsed)

    return labels


# convert yolo format from labels, to pixels
def yolo_to_coords(box, img_w, img_h):
    # source: https://www.geeksforgeeks.org/python/python-pil-image-crop-method/
    # yolo format is (x_center, y_center, width, height) normalized to [0,1]
    # .crop() requires (left, top, right, bottom)
    x, y, w, h = box

    left = int((x - w / 2) * img_w) # * img_w to get pixel coord from normalized
    top = int((y - h / 2) * img_h)
    right = int((x + w / 2) * img_w)
    bottom = int((y + h / 2) * img_h)

    return left, top, right, bottom


# embeded crop using dinov2
def embed_crop(crop_img):
    inputs = processor(images=crop_img, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    emb = out.last_hidden_state[:, 0, :].squeeze()
    return emb.cpu().numpy()


# process Roboflow dataset
embeddings = []
labels = []
sources = []

image_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

print(f"Found {len(image_files)} images")

print("\nExtracting object embeddings\n")

# run for every image
for fname in tqdm(image_files):
    img_path = os.path.join(IMG_DIR, fname)

    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
    except:
        continue

    img_w, img_h = img.size

    yolo_labels = load_labels(fname)

    for class_id, box in yolo_labels:
        # get pixel coords from yolo format labels
        left, top, right, bottom = yolo_to_coords(box, img_w, img_h)

        x1, y1 = max(0, left), max(0, top)
        x2, y2 = min(img_w, right), min(img_h, bottom)

        if x2 <= x1 or y2 <= y1:
            continue

        # crop the image to the bounding box of the object
        crop = img.crop((x1, y1, x2, y2))

        # get embedding for the cropped object
        emb = embed_crop(crop)

        embeddings.append(emb)
        labels.append(class_id)
        sources.append(fname)
    
    if len(embeddings) >= 1000:
        emb_array = np.stack(embeddings)
        emb_array /= (np.linalg.norm(emb_array, axis=1, keepdims=True) + 1e-8) # normalize embeddings to unit length for cosine similarity

        np.savez(f"chunk_{chunk_id}.npz",
                embeddings=emb_array,
                labels=np.array(labels),
                sources=np.array(sources))

        embeddings.clear()
        labels.clear()
        sources.clear()
        chunk_id += 1

if len(embeddings) > 0:
    emb_array = np.stack(embeddings)
    emb_array /= (np.linalg.norm(emb_array, axis=1, keepdims=True) + 1e-8)

    np.savez(
        f"chunk_{chunk_id}.npz",
        embeddings=emb_array,
        labels=np.array(labels),
        sources=np.array(sources)
    )

print(f"Saved all chunks")

### join chunks into a single file

In [ ]:
# path to chunks
DATASET_DIR = "./ingredients_data" # dataset with images and labels (YOLOv8 format)
SPLIT = "train"

chunk_dir = f"{DATASET_DIR}/{SPLIT}_chunks"
files = sorted(glob.glob(f"{chunk_dir}/*.npz"))

all_embeddings = []
all_labels = []
all_sources = []

for f in files:
    data = np.load(f)
    all_embeddings.append(data["embeddings"])
    all_labels.append(data["labels"])
    all_sources.append(data["sources"])

embeddings = np.concatenate(all_embeddings, axis=0)
labels = np.concatenate(all_labels, axis=0)
sources = np.concatenate(all_sources, axis=0)

np.savez(
    f"dinov2_objects_{SPLIT}.npz",
    embeddings=embeddings,
    labels=labels,
    sources=sources
)

print("Merged successfully")